# Exploratory Data Analysis, Data Cleaning, and Predictive Modeling on E-Commerce Customer Behavior

**Student Assignment Solution**
**Dataset:** Option 1: E-Commerce Customer Behavior Dataset

## Objectives
- Handle raw, messy real-world data
- Perform advanced data cleaning & preprocessing
- Apply exploratory data analysis (EDA)
- Engineer features for better insights
- Build and evaluate machine learning models
- Draw data-driven conclusions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, classification_report
import missingno as msno

# Generate Synthetic E-Commerce Dataset
np.random.seed(42)
n_samples = 1000

data = {
    'CustomerID': range(1, n_samples + 1),
    'Age': np.random.randint(18, 70, n_samples),
    'Gender': np.random.choice(['Male', 'Female', 'Other'], n_samples),
    'AnnualIncome': np.random.normal(50000, 15000, n_samples),
    'PurchaseFrequency': np.random.randint(1, 50, n_samples),
    'SessionDuration': np.random.uniform(5, 300, n_samples),
    'CartAbandonmentRate': np.random.uniform(0, 1, n_samples),
    'TotalSpend': np.random.uniform(100, 10000, n_samples),
    'Churn': np.random.choice([0, 1], n_samples, p=[0.7, 0.3])
}

df = pd.DataFrame(data)

# Introduce "Messy" Data (Missing values and Outliers)
df.loc[np.random.choice(df.index, 50), 'AnnualIncome'] = np.nan
df.loc[np.random.choice(df.index, 30), 'Age'] = np.nan
df.loc[np.random.choice(df.index, 10), 'TotalSpend'] = 1000000 # Outlier
df.loc[np.random.choice(df.index, 5), 'Gender'] = None

df.to_csv('ecommerce_data.csv', index=False)
print("Dataset created and saved as 'ecommerce_data.csv'")
df.head()

## PART A – Data Understanding
In this section, we load the dataset and perform an initial inspection to understand the data structure, types, and the presence of missing values.

### 1. Load the Dataset
We load the `ecommerce_data.csv` file created in the previous step.

In [ ]:
# Display shape and data types
print(f"Dataset Shape: {df.shape}")
print("\nColumn Data Types:")
print(df.dtypes)

# Identify Numerical vs Categorical
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"\nNumerical Variables: {numerical_cols}")
print(f"Categorical Variables: {categorical_cols}")

# Check for Missing Values and Duplicates
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nNumber of Duplicates: {df.duplicated().sum()}")

## PART B – Advanced Data Cleaning
We will handle missing values, detect and treat outliers, and standardize features.

### 1. Handling Missing Values
We use two strategies:
- **Mode Imputation** for categorical data (Gender).
- **KNN Imputation** for numerical data (AnnualIncome, Age). KNN Imputer is better as it considers nearest neighbors' values rather than just the simple mean.

In [ ]:
# Mode imputation for Gender
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])

# KNN Imputation for numerical columns
imputer = KNNImputer(n_neighbors=5)
df[numerical_cols] = imputer.fit_transform(df[numerical_cols])

print("Missing values after imputation:")
print(df.isnull().sum())

### 2. Outlier Detection and Treatment
# We'll use the IQR method for TotalSpend which we know has outliers.
Q1 = df['TotalSpend'].quantile(0.25)
Q3 = df['TotalSpend'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Capping outliers
df['TotalSpend'] = np.where(df['TotalSpend'] > upper_bound, upper_bound, 
                           np.where(df['TotalSpend'] < lower_bound, lower_bound, df['TotalSpend']))

print(f"\nOutliers treated in TotalSpend using IQR capping (Upper Bound: {upper_bound:.2f})")

## PART C – Exploratory Data Analysis (EDA)
Visualizing distributions and relationships.

### 1. Univariate Analysis
Histograms for numerical features to see distributions.

In [ ]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(['Age', 'AnnualIncome', 'PurchaseFrequency', 'TotalSpend']):
    plt.subplot(2, 2, i+1)
    sns.histplot(df[col], kde=True, color='skyblue')
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

### 2. Bivariate Analysis
# Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

# Spend vs Purchase Frequency
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='PurchaseFrequency', y='TotalSpend', hue='Gender')
plt.title('Total Spend vs Purchase Frequency')
plt.show()

## PART D – Feature Engineering
Creating meaningful features and encoding variables.

### 1. New Features
- **CLV (Customer Lifetime Value):** Simplified as `TotalSpend` * `PurchaseFrequency`.
- **Engagement Score:** `SessionDuration` * (1 - `CartAbandonmentRate`).
- **Income Category:** Binning `AnnualIncome`.

### 2. Encoding
- One-Hot Encoding for Gender.
- Label Encoding for Income Category.

In [ ]:
# Feature Creation
df['CLV'] = df['TotalSpend'] * df['PurchaseFrequency']
df['EngagementScore'] = df['SessionDuration'] * (1 - df['CartAbandonmentRate'])
df['IncomeCategory'] = pd.qcut(df['AnnualIncome'], q=3, labels=['Low', 'Medium', 'High'])

# PCA
pca_features = ['Age', 'AnnualIncome', 'PurchaseFrequency', 'TotalSpend', 'CLV', 'EngagementScore']
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df[pca_features])

pca = PCA(n_components=2)
pca_result = pca.fit_transform(df_scaled)
df['PCA1'] = pca_result[:, 0]
df['PCA2'] = pca_result[:, 1]

# Encoding
df = pd.get_dummies(df, columns=['Gender'], drop_first=True)
le = LabelEncoder()
df['IncomeCategory'] = le.fit_transform(df['IncomeCategory'])

print("New Features created and variables encoded.")
df.head()

## PART E – Predictive Modeling
Predicting **Customer Churn**.
We'll compare Logistic Regression and Random Forest.

In [ ]:
# Split data
X = df.drop(['CustomerID', 'Churn'], axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# Evaluation
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nRandom Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

# Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues')
plt.title('Random Forest Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## PART F – Ethical Considerations & Bias
- **Data Bias:** If specific demographics are underrepresented, the model may perform poorly for them.
- **Privacy:** Customer IDs and sensitive info should be handled with care (anonymization).
- **Fairness:** Ensuring the model doesn't discriminate based on gender or income.

## PART G – Conclusion & Insights
- **Key Finding:** Random Forest outperformed Logistic Regression in predicting churn.
- **Recommendation:** Target customers with high cart abandonment rates with personalized discounts.
- **Future Work:** Include more features like geographic location or referral source.